# RO3D — Eikonal Weight Ablation

Sweeps the eikonal loss weight over six values and three random seeds
(18 training runs) to quantify how sensitive the method is to this
hyperparameter, and to select the value used in `02_train_and_evaluate.ipynb`.

**Runtime:** ~3.3 h on a T4 (18 runs x ~11 min). Results are appended to CSV
after every run and completed runs are skipped on restart, so a disconnect
costs at most one run.

**Why the control matters:** `w_eik = 0` removes the signed-distance
constraint entirely. Solid-body rotation is a rigid motion, so the exact
solution stays an exact SDF for all time and the eikonal term is consistent
with the transport equation here — this benchmark is favourable to it, which
is worth stating when generalising the selected value to deformation cases.

Note that the eikonal weight selected here need not match the one selected on
the translation benchmark; reporting both is more informative than assuming
one transfers.

**Outputs:** `RO3D_eikonal_ablation.csv` (per-snapshot and averaged metrics for
every run) and a summary table with seed spread.

In [ ]:
# --- setup -------------------------------------------------------------
import sys, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/RO3D'
except ImportError:
    PROJECT_DIR = os.getcwd()

sys.path.insert(0, PROJECT_DIR)
os.makedirs(PROJECT_DIR, exist_ok=True)

import csv, itertools, time
import numpy as np
import torch
import ro3d_core as core

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

Mounted at /content/drive
device: cuda


In [ ]:
# --- configuration -----------------------------------------------------
W_EIK_GRID   = [0.0, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]
SEEDS        = [42, 43, 44]

W_PDE, W_IC  = 1.0, 10.0
N_F, N_I     = 10000, 5000
LAYERS, NEURONS = 8, 256
T_FINAL      = 2 * np.pi   # one full revolution
ADAM_EPOCHS  = 20000        # same budget as the reported run, so the ranking
LBFGS_EPOCHS = 500          # transfers without extrapolation
GRID_N       = 201
N_SNAPSHOTS  = 6

CSV_PATH = f'{PROJECT_DIR}/RO3D_eikonal_ablation.csv'

times, tlabels = core.snapshot_times(T_FINAL, N_SNAPSHOTS)
grid = core.Grid(n=GRID_N)
METRICS = ['rel_L2', 'vol_MAPE', 'eik_dev', 'l2_abs', 'vol_abs']

FIELDS = ['w_eik', 'seed']
for m in METRICS:
    FIELDS += [f'{m}_t{i}' for i in range(N_SNAPSHOTS)] + [f'avg_{m}']
FIELDS += ['loss_adam', 'loss_final', 'lbfgs_gain', 'minutes']

print(f'{len(W_EIK_GRID) * len(SEEDS)} runs, {len(FIELDS)} columns per row')

18 runs, 41 columns per row


In [ ]:
# --- one run -----------------------------------------------------------
def run_one(w_eik, seed):
    net = core.build_model(device, seed=seed, layers=LAYERS, neurons=NEURONS)
    X_f, X_i = core.sample_points(N_F, N_I, device, seed=seed)
    loss_fn = core.make_loss(net, X_f, X_i, T_FINAL, W_PDE, W_IC, w_eik)

    tr = core.train(net, loss_fn, adam_epochs=ADAM_EPOCHS,
                    lbfgs_epochs=LBFGS_EPOCHS, verbose=False)
    res = core.evaluate(net, grid, times, T_FINAL, device, cache=False)

    row = {'w_eik': w_eik, 'seed': seed}
    for m in METRICS:
        for i in range(N_SNAPSHOTS):
            row[f'{m}_t{i}'] = f'{res[m][i]:.6e}'
        row[f'avg_{m}'] = f"{res['avg_' + m]:.6e}"
    row.update({'loss_adam': f"{tr['loss_adam']:.6e}",
                'loss_final': f"{tr['loss_final']:.6e}",
                'lbfgs_gain': f"{tr['lbfgs_gain']:.6e}",
                'minutes': f"{tr['minutes']:.2f}"})

    del net, X_f, X_i
    if device == 'cuda':
        torch.cuda.empty_cache()
    return row

In [4]:
# --- sweep (resumable) -------------------------------------------------
done = set()
if os.path.exists(CSV_PATH):
    with open(CSV_PATH) as f:
        for r in csv.DictReader(f):
            done.add((float(r['w_eik']), int(r['seed'])))
    print(f'{len(done)} run(s) already complete — skipping those')

todo = [p for p in itertools.product(W_EIK_GRID, SEEDS) if p not in done]
print(f'{len(todo)} run(s) to go, ~{len(todo) * 11 / 60:.1f} h\n', flush=True)

for k, (wv, sd) in enumerate(todo, 1):
    print(f'[{k}/{len(todo)}] w_eik={wv:g} seed={sd} ...', flush=True)
    row = run_one(wv, sd)
    new = not os.path.exists(CSV_PATH) or os.path.getsize(CSV_PATH) == 0
    with open(CSV_PATH, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if new:
            w.writeheader()
        w.writerow(row)
    print(f"    relL2={float(row['avg_rel_L2']):.4f}%  "
          f"MAPE={float(row['avg_vol_MAPE']):.4f}%  "
          f"eik={float(row['avg_eik_dev']):.3e}  "
          f"lbfgs_gain={float(row['lbfgs_gain']):.2e}  "
          f"({float(row['minutes']):.1f} min)", flush=True)

1 run(s) already complete — skipping those
17 run(s) to go, ~3.1 h

[1/17] w_eik=0 seed=43 ...


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    relL2=0.4256%  MAPE=1.2009%  eik=7.040e-02  lbfgs_gain=1.01e-04  (16.4 min)
[2/17] w_eik=0 seed=44 ...
    relL2=1.5804%  MAPE=27.5388%  eik=2.816e-01  lbfgs_gain=7.52e-09  (12.1 min)
[3/17] w_eik=0.0001 seed=42 ...
    relL2=1.6782%  MAPE=29.0365%  eik=2.822e-01  lbfgs_gain=2.04e-08  (12.1 min)
[4/17] w_eik=0.0001 seed=43 ...
    relL2=1.6683%  MAPE=32.4515%  eik=2.952e-01  lbfgs_gain=2.01e-08  (12.1 min)
[5/17] w_eik=0.0001 seed=44 ...
    relL2=1.5723%  MAPE=30.1003%  eik=2.949e-01  lbfgs_gain=5.53e-09  (12.1 min)
[6/17] w_eik=0.001 seed=42 ...
    relL2=1.6186%  MAPE=32.1018%  eik=3.007e-01  lbfgs_gain=6.46e-08  (12.1 min)
[7/17] w_eik=0.001 seed=43 ...
    relL2=1.5809%  MAPE=31.1745%  eik=2.939e-01  lbfgs_gain=1.08e-08  (12.1 min)
[8/17] w_eik=0.001 seed=44 ...
    relL2=0.4192%  MAPE=1.3479%  eik=6.862e-02  lbfgs_gain=1.03e-04  (16.4 min)
[9/17] w_eik=0.01 seed=42 ...
    relL2=1.2417%  MAPE=25.2681%  eik=2.600e-01  lbfgs_gain=4.83e-08  (12.1 min)
[10/17] w_eik=0.01 seed=43 

## Summary and selection

Decision rule, fixed before the sweep:

1. **Primary** — lowest average relative L₂
2. **Tiebreak 1** — lowest average volume MAPE
3. **Tiebreak 2** — lowest eikonal deviation
4. **Tiebreak 3** — the smaller weight

Two weights count as *tied* when the gap between their means is smaller than
the larger of their two seed standard deviations. Eikonal deviation is only a
second-order tiebreak on purpose: it is essentially the quantity the swept term
minimises, so ranking by it would recover the sweep by construction.

In [5]:
# --- summary -----------------------------------------------------------
rows = list(csv.DictReader(open(CSV_PATH)))
agg = {}
for r in rows:
    agg.setdefault(float(r['w_eik']), []).append(r)

stats = {}
print(f"{'w_eik':>8} | {'rel L2 (%)':>19} | {'vol MAPE (%)':>19} | "
      f"{'eik dev':>11} | {'n':>2}")
print('-' * 74)
for wv in sorted(agg):
    g = agg[wv]
    m = {k: np.mean([float(r['avg_' + k]) for r in g]) for k in
         ('rel_L2', 'vol_MAPE', 'eik_dev')}
    s = {k: np.std([float(r['avg_' + k]) for r in g]) for k in
         ('rel_L2', 'vol_MAPE', 'eik_dev')}
    stats[wv] = (m, s, len(g))
    print(f"{wv:>8g} | {m['rel_L2']:>9.4f} ± {s['rel_L2']:<7.4f} | "
          f"{m['vol_MAPE']:>9.4f} ± {s['vol_MAPE']:<7.4f} | "
          f"{m['eik_dev']:>11.3e} | {len(g):>2}")

def tied(a, b, key):
    (ma, sa, _), (mb, sb, _) = stats[a], stats[b]
    return abs(ma[key] - mb[key]) < max(sa[key], sb[key])

best = min(stats, key=lambda w: stats[w][0]['rel_L2'])
for cand in sorted(stats, key=lambda w: stats[w][0]['rel_L2'])[1:]:
    if tied(best, cand, 'rel_L2'):
        if not tied(best, cand, 'vol_MAPE'):
            best = min(best, cand, key=lambda w: stats[w][0]['vol_MAPE'])
        elif not tied(best, cand, 'eik_dev'):
            best = min(best, cand, key=lambda w: stats[w][0]['eik_dev'])
        else:
            best = min(best, cand)

ties = sorted(w for w in stats if w != best and tied(best, w, 'rel_L2'))
print('-' * 74)
print(f'selected w_eik = {best:g}')
if ties:
    print('statistically tied on rel L2 with: ' + ', '.join(f'{w:g}' for w in ties))
    print('-> report insensitivity across this range rather than a single value')
print(f"mean L-BFGS gain over all runs: "
      f"{np.mean([float(r['lbfgs_gain']) for r in rows]):.3e}")

   w_eik |          rel L2 (%) |        vol MAPE (%) |     eik dev |  n
--------------------------------------------------------------------------
       0 |    1.2570 ± 0.5927  |   20.1083 ± 13.4712 |   2.155e-01 |  3
  0.0001 |    1.6396 ± 0.0477  |   30.5295 ± 1.4268  |   2.908e-01 |  3
   0.001 |    1.2062 ± 0.5567  |   21.5414 ± 14.2840 |   2.211e-01 |  3
    0.01 |    1.0965 ± 0.1083  |   19.4596 ± 4.1285  |   2.347e-01 |  3
     0.1 |    0.2967 ± 0.2053  |    0.9127 ± 0.9839  |   2.768e-02 |  3
       1 |    1.1814 ± 0.6984  |    4.5202 ± 2.5354  |   2.643e-02 |  3
--------------------------------------------------------------------------
selected w_eik = 0.1
mean L-BFGS gain over all runs: 4.894e-05


## Error growth per snapshot

Per-snapshot averages across seeds, for the time-evolution table/figure.

In [6]:
# --- per-snapshot table ------------------------------------------------
METRIC = 'rel_L2'      # 'rel_L2' | 'vol_MAPE' | 'eik_dev' | 'l2_abs'

hdr = ' | '.join(f'{lab:>9}' for lab in tlabels)
print(f'{METRIC} by snapshot (mean over seeds)\n')
print(f"{'w_eik':>8} | {hdr}")
print('-' * (11 + 12 * len(tlabels)))
for wv in sorted(agg):
    vals = [np.mean([float(r[f'{METRIC}_t{i}']) for r in agg[wv]])
            for i in range(N_SNAPSHOTS)]
    print(f'{wv:>8g} | ' + ' | '.join(f'{v:>9.4f}' for v in vals))

rel_L2 by snapshot (mean over seeds)

   w_eik |         0 |       T/5 |      2T/5 |      3T/5 |      4T/5 |         T
-----------------------------------------------------------------------------------
       0 |    0.4512 |    0.9205 |    1.3107 |    1.5191 |    1.6166 |    1.7238
  0.0001 |    0.6195 |    1.2147 |    1.6389 |    1.9680 |    2.1457 |    2.2508
   0.001 |    0.4776 |    0.9088 |    1.2194 |    1.4820 |    1.5326 |    1.6171
    0.01 |    0.5167 |    0.8483 |    1.0624 |    1.2528 |    1.3921 |    1.5064
     0.1 |    0.1572 |    0.2305 |    0.2842 |    0.3366 |    0.3627 |    0.4091
       1 |    0.3045 |    0.7960 |    1.1663 |    1.3972 |    1.6378 |    1.7866


In [7]:
# --- per-snapshot table ------------------------------------------------
METRIC = 'vol_MAPE'      # 'rel_L2' | 'vol_MAPE' | 'eik_dev' | 'l2_abs'

hdr = ' | '.join(f'{lab:>9}' for lab in tlabels)
print(f'{METRIC} by snapshot (mean over seeds)\n')
print(f"{'w_eik':>8} | {hdr}")
print('-' * (11 + 12 * len(tlabels)))
for wv in sorted(agg):
    vals = [np.mean([float(r[f'{METRIC}_t{i}']) for r in agg[wv]])
            for i in range(N_SNAPSHOTS)]
    print(f'{wv:>8g} | ' + ' | '.join(f'{v:>9.4f}' for v in vals))

vol_MAPE by snapshot (mean over seeds)

   w_eik |         0 |       T/5 |      2T/5 |      3T/5 |      4T/5 |         T
-----------------------------------------------------------------------------------
       0 |    3.8239 |   12.4944 |   19.4786 |   24.8348 |   28.4401 |   31.5781
  0.0001 |    5.0659 |   18.4800 |   28.7213 |   38.0015 |   44.4452 |   48.4630
   0.001 |    4.6153 |   14.1042 |   21.0758 |   26.8719 |   30.3000 |   32.2814
    0.01 |    1.2783 |   11.4710 |   18.3559 |   24.4138 |   28.8719 |   32.3668
     0.1 |    1.0842 |    1.0648 |    1.1971 |    1.1390 |    0.7087 |    0.2826
       1 |    1.5821 |    3.2792 |    5.2677 |    5.6187 |    6.0888 |    5.2845


In [8]:
# --- per-snapshot table ------------------------------------------------
METRIC = 'eik_dev'      # 'rel_L2' | 'vol_MAPE' | 'eik_dev' | 'l2_abs'

hdr = ' | '.join(f'{lab:>9}' for lab in tlabels)
print(f'{METRIC} by snapshot (mean over seeds)\n')
print(f"{'w_eik':>8} | {hdr}")
print('-' * (11 + 12 * len(tlabels)))
for wv in sorted(agg):
    vals = [np.mean([float(r[f'{METRIC}_t{i}']) for r in agg[wv]])
            for i in range(N_SNAPSHOTS)]
    print(f'{wv:>8g} | ' + ' | '.join(f'{v:>9.4f}' for v in vals))

eik_dev by snapshot (mean over seeds)

   w_eik |         0 |       T/5 |      2T/5 |      3T/5 |      4T/5 |         T
-----------------------------------------------------------------------------------
       0 |    0.1119 |    0.1762 |    0.2162 |    0.2465 |    0.2649 |    0.2771
  0.0001 |    0.1693 |    0.2420 |    0.2896 |    0.3287 |    0.3514 |    0.3638
   0.001 |    0.1188 |    0.1863 |    0.2244 |    0.2523 |    0.2671 |    0.2775
    0.01 |    0.1301 |    0.2005 |    0.2350 |    0.2645 |    0.2815 |    0.2965
     0.1 |    0.0178 |    0.0225 |    0.0231 |    0.0254 |    0.0324 |    0.0450
       1 |    0.0286 |    0.0248 |    0.0192 |    0.0238 |    0.0251 |    0.0371


In [9]:
# --- per-snapshot table ------------------------------------------------
METRIC = 'l2_abs'      # 'rel_L2' | 'vol_MAPE' | 'eik_dev' | 'l2_abs'

hdr = ' | '.join(f'{lab:>9}' for lab in tlabels)
print(f'{METRIC} by snapshot (mean over seeds)\n')
print(f"{'w_eik':>8} | {hdr}")
print('-' * (11 + 12 * len(tlabels)))
for wv in sorted(agg):
    vals = [np.mean([float(r[f'{METRIC}_t{i}']) for r in agg[wv]])
            for i in range(N_SNAPSHOTS)]
    print(f'{wv:>8g} | ' + ' | '.join(f'{v:>9.4f}' for v in vals))

l2_abs by snapshot (mean over seeds)

   w_eik |         0 |       T/5 |      2T/5 |      3T/5 |      4T/5 |         T
-----------------------------------------------------------------------------------
       0 |    0.0019 |    0.0039 |    0.0055 |    0.0064 |    0.0068 |    0.0073
  0.0001 |    0.0026 |    0.0051 |    0.0069 |    0.0083 |    0.0091 |    0.0095
   0.001 |    0.0020 |    0.0038 |    0.0051 |    0.0063 |    0.0065 |    0.0068
    0.01 |    0.0022 |    0.0036 |    0.0045 |    0.0053 |    0.0059 |    0.0064
     0.1 |    0.0007 |    0.0010 |    0.0012 |    0.0014 |    0.0015 |    0.0017
       1 |    0.0013 |    0.0034 |    0.0049 |    0.0059 |    0.0069 |    0.0075
